# Misinformation Models — Sprint 3: Multi-Task Learning
Driver notebook: runs all Sprint 2 models as baselines, adds LogReg MTL (cascaded POS).
CNN cells commented out. Train/dev only — test set not touched.
Had help from Claude on implementation.

### Import configuration and packages ###

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
# import torch                       # CNN only
# import cnn_baseline as cnn         # CNN only

# Running this will import FastText vector file, stored on HuggingFace and is >4gb.
from config import DATA_DIR, FASTTEXT_PATH, TARGETS

# from preprocess import preprocess  # CNN only
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report

# DEVICE = torch.device(             # CNN only
#     "mps"  if torch.backends.mps.is_available()  else
#     "cuda" if torch.cuda.is_available()           else
#     "cpu"
# )
# print(f"Device: {DEVICE}")         # CNN only
print(f"Targets: {TARGETS}")

/home/nicole-shantz/miniforge3/envs/colx_523/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Targets: ['opinion_label', 'misinformation_label']


In [2]:
# Load data (shared across all models)
import logreg_transfer as lr_transfer
train_rows = lr_transfer.load_csv("mis_df_train.csv")
dev_rows   = lr_transfer.load_csv("mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


## Run models


In [ ]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {} 

#### CNN Baseline (commented out)


In [4]:
# CNN Baseline — commented out pending CNN MTL implementation
# vocab        = cnn.build_vocab(train_rows, preprocess)
# embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
# train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
# dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
# for target in TARGETS:
#     model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
#     model = cnn.train_model(model, train_loader, dev_loader, train_rows, DEVICE,
#                             train_targets=[target])
#     results[f"TextCNN — {target}"] = cnn.predict(model, dev_loader, DEVICE, target=target)

#### CNN Transfer Learning (commented out)


In [5]:
# CNN Transfer Learning — commented out pending CNN MTL implementation
# cnn_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
# cnn_model = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)
# for target in TARGETS:
#     results[f"TextCNN Transfer — {target}"] = cnn.predict(cnn_model, dev_loader, DEVICE, target=target)

#### Logistic Regression Baseline

In [6]:
# Run Logistic Regression model — trained separately per target
import logreg_baseline as lr

for target in TARGETS:
    results[f"LogReg — {target}"] = lr.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7013

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8514


#### Logistic Regression Transfer Learning

In [7]:
# Run Logistic Regression transfer model — trained separately per target
import logreg_transfer as lr_transfer

for target in TARGETS:
    results[f"LogReg (+ embeddings) — {target}"] = lr_transfer.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7285

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8735


#### LogReg MTL — Cascaded POS Prediction (Sprint 3)


In [ ]:
# Sprint 3 MTL: Logistic Regression with cascaded POS distribution features
import logreg_mtl as lr_mtl

for target in TARGETS:
    results[f"LogReg MTL (cascaded POS) — {target}"] = lr_mtl.run(train_rows, dev_rows, task=target)


── LogReg MTL Cascaded POS  [opinion_label] ──
  Building features (TF-IDF, FastText, linguistic, cascaded POS)
Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 tokens found in FastText (81.5%)
  Running secondary task: POS tagging 600 documents …
  POS distribution feature dim: 15 tags
  Running secondary task: POS tagging 200 documents …
  POS distribution feature dim: 15 tags
  Feature matrix: train=(600, 2819), dev=(200, 2819)
  Running GridSearchCV over C


/home/nicole-shantz/miniforge3/envs/colx_523/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Best C: 0.1  |  CV macro-F1: 0.7169

── LogReg MTL Cascaded POS  [misinformation_label] ──
  Building features (TF-IDF, FastText, linguistic, cascaded POS)
Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 tokens found in FastText (81.5%)
  Running secondary task: POS tagging 600 documents …
  POS distribution feature dim: 15 tags
  Running secondary task: POS tagging 200 documents …
  POS distribution feature dim: 15 tags
  Feature matrix: train=(600, 2819), dev=(200, 2819)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8480


#### Results

In [9]:
#Create table to compare models across metrics
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Macro F1": m["macro_f1"],
                 "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                 "F1.5 (recall-weighted)": m["fbeta_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
LogReg — opinion_label,0.6931,0.7391,0.6471,0.6530,0.7514
LogReg — misinformation_label,0.8784,0.9404,0.8163,0.7975,0.9630
LogReg (+ embeddings) — opinion_label,0.7023,0.7306,0.6740,0.6962,0.7705
LogReg (+ embeddings) — misinformation_label,0.8889,0.9428,0.8350,0.8318,0.9502
LogReg MTL (cascaded POS) — opinion_label,0.6963,0.7297,0.6629,0.6806,0.7488
LogReg MTL (cascaded POS) — misinformation_label,0.8831,0.9392,0.8269,0.8269,0.9150


In [10]:
#Model details
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


LogReg — opinion_label
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    85      32
  true=1 (opinion):   28      55

              precision    recall  f1-score   support

 not-opinion       0.75      0.73      0.74       117
     opinion       0.63      0.66      0.65        83

    accuracy                           0.70       200
   macro avg       0.69      0.69      0.69       200
weighted avg       0.70      0.70      0.70       200


False Positives (predicted opinion, actually not) — 5 shown:
  [8] "No food, no FEMA: Hurricane Michael's survivors are furious - The Daily Beast https://apple.news/APp4E5UMtQT2ULJVMPM0ovw"
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTennis 

### Simple Ensembling

In [ ]:
# Soft vote ensemble (CNN commented out)
import importlib
import simple_ensemble
importlib.reload(simple_ensemble)
from simple_ensemble import soft_vote

ensemble_rows = []
for task in TARGETS:
    preds, labels, probs = soft_vote([
        results[f"LogReg — {task}"],
        results[f"LogReg (+ embeddings) — {task}"],
        results[f"LogReg MTL (cascaded POS) — {task}"],
        # results[f"TextCNN — {task}"],           # CNN only
        # results[f"TextCNN Transfer — {task}"],   # CNN only
    ])
    m = compute_metrics(preds, labels, probs)
    ensemble_rows.append({"Model": f"Soft Vote Ensemble — {task}",
                          "Macro F1": m["macro_f1"],
                          "F1 (not-op)": m["f1_class0"],
                          "F1 (opinion)": m["f1_class1"],
                          "F1.5 (recall-weighted)": m["fbeta_class1"],
                          "AUC-ROC": m["auc_roc"]})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(ensemble_rows).set_index("Model")

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Soft Vote Ensemble — opinion_label,0.6855,0.7232,0.6477,0.6622,0.7685
Soft Vote Ensemble — misinformation_label,0.8947,0.9463,0.8431,0.8368,0.9519


### Motivated Ensembling

In [ ]:
# Motivated (F1-weighted) ensemble (CNN commented out)
import importlib
import motivated_ensemble
importlib.reload(motivated_ensemble)
from motivated_ensemble import motivated_soft_vote

f1_scores = {
    "opinion_label":        [0.6931, 0.7029, 0.7023],
    "misinformation_label": [0.8784, 0.8889, 0.8889],
}

for task, weights in f1_scores.items():
    print(f"{task}: {weights}, all positive: {all(w > 0 for w in weights)}")

motivated_rows = []
for task in TARGETS:
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[
            results[f"LogReg — {task}"],
            results[f"LogReg (+ embeddings) — {task}"],
            results[f"LogReg MTL (cascaded POS) — {task}"],
            # results[f"TextCNN — {task}"],           # CNN only
            # results[f"TextCNN Transfer — {task}"],   # CNN only
        ],
        f1_weights=f1_scores[task],
    )
    m = compute_metrics(preds, labels, probs)
    motivated_rows.append({"Model": f"Motivated Ensemble — {task}",
                           "Macro F1": m["macro_f1"],
                           "F1 (not-op)": m["f1_class0"],
                           "F1 (opinion)": m["f1_class1"],
                           "F1.5 (recall-weighted)": m["fbeta_class1"],
                           "AUC-ROC": m["auc_roc"]})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(motivated_rows).set_index("Model")

opinion_label: [0.6931, 0.7029, 0.7023], all positive: True
misinformation_label: [0.8784, 0.8889, 0.8889], all positive: True

  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.6347      0.5909      0.6786      0.6400
        0.35      0.6732      0.6486      0.6977      0.6750
        0.40      0.7148      0.7077      0.7220      0.7150
        0.45      0.7247      0.7343      0.7150      0.7250 ◄
        0.50      0.6855      0.7232      0.6477      0.6900
        0.55      0.6911      0.7436      0.6386      0.7000
        0.60      0.6376      0.7373      0.5379      0.6650
        0.65      0.5713      0.7299      0.4127      0.6300
        0.70      0.5386      0.7324      0.3448      0.6200

  Selected threshold: 0.45

  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ---------------------------

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Motivated Ensemble — opinion_label,0.7247,0.7343,0.7150,0.7557,0.7686
Motivated Ensemble — misinformation_label,0.8960,0.9459,0.8462,0.8462,0.9518
